# DRR Generation (DiffDRR) — Bi-planar AP/LAT X-rays [HPC]

HPC version of `notebooks/pre-processing/drr_generation.ipynb`.
Render AP + LAT DRRs from all preprocessed CT volumes in `data/interim/predrr/` using
**DiffDRR on GPU**. Outputs land in `data/interim/DRRs/`.

**Key differences from local notebook:**
- `ENV = "HPC"`, `DEVICE = "cuda"` — batch loop runs by default
- Paths are Linux, rooted at the HPC working directory
- Includes a quick single-case smoke test (Cell 9) before the full batch

## Environment setup (run once in terminal before launching)
```bash
cd "/home/project/xray2mesh/Marcus_Chan_Zheng_Shao_CP2 _24020059"
source .venv/bin/activate
pip install -r requirements_hpc.txt   # includes diffdrr==0.6.1, torchio==1.2.0
```

## Input
256³ NIfTI, 0.78125 mm isotropic, 200 mm FOV, **LPS** orientation, float [0,1]
(bone-windowed at [-450, 1050] HU, normalized). Already knee-cropped + split per side.
- Healthy: 58 files / 30 cases (`VSD_<id>_Left/Right.nii.gz`)
- Fractured: 14 files / 14 cases (`Case<N>_PartLeft/Right.nii.gz`)

## Output
```
data/interim/DRRs/
├── healthy/   VSD_001/{left,right}/{ap,lat}.{png,npy}
└── fractured/ Case1/{left}/{ap,lat}.{png,npy}
```

## Validated geometry (do not change without re-testing)
| Parameter | Value |
|-----------|-------|
| SDD | 1000 mm |
| SOD | 850 mm (~1.18× magnification) |
| Resolution | 256×256 |
| delx / dely | 1.4 mm |
| AP pose | rot [0,0,0], ZXY degrees |
| LAT pose | rot [90,0,0], ZXY degrees |
| Beer-Lambert mu | 2.0 |
| bone_attenuation_multiplier | 3.0 |

In [ ]:
import numpy as np
import pandas as pd
import nibabel as nib
import torch
import imageio.v2 as imageio
import matplotlib.pyplot as plt
from pathlib import Path
from torchio import ScalarImage
from loguru import logger
from tqdm import tqdm

import diffdrr
from diffdrr.drr import DRR
from diffdrr.data import read

# Developed against diffdrr 0.6.1
print("diffdrr", diffdrr.__version__, "| torch", torch.__version__, "| cuda?", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ---- Configuration (HPC) ----
# Note the space before underscore in the HPC working directory name
PROJECT_ROOT = Path("/home/project/xray2mesh/Marcus_Chan_Zheng_Shao_CP2 _24020059")
PREDRR_DIR   = PROJECT_ROOT / "data" / "interim" / "predrr"
DRR_OUT_DIR  = PROJECT_ROOT / "data" / "interim" / "DRRs"

ENV    = "HPC"    # activates batch loop
DEVICE = "cuda"   # GPU

# Geometry (validated locally — do not change without visual re-testing)
SDD    = 1000.0
SOD    = 850.0
HEIGHT = 256
WIDTH  = 256
DELX   = 1.4
DELY   = 1.4

# Density / intensity
HU_WINDOW = (-450.0, 1050.0)
BONE_MULT = 3.0
MU        = 2.0

# AP / LAT poses (Euler angles, degrees, ZXY convention)
VIEWS = {
    "ap":  {"rotation": [0.0, 0.0, 0.0],  "translation": [0.0, SOD, 0.0]},
    "lat": {"rotation": [90.0, 0.0, 0.0], "translation": [0.0, SOD, 0.0]},
}

DRR_OUT_DIR.mkdir(parents=True, exist_ok=True)
logger.info(f"ENV={ENV} DEVICE={DEVICE} | out -> {DRR_OUT_DIR}")
logger.info(f"predrr healthy: {len(list((PREDRR_DIR / 'healthy').glob('*.nii.gz')))} files")
logger.info(f"predrr fractured: {len(list((PREDRR_DIR / 'fractured').glob('*.nii.gz')))} files")

In [ ]:
def parse_case_side(filename: str):
    """Map a predrr filename to (case_id, side).

    'VSD_001_Left.nii.gz'    -> ('VSD_001', 'left')
    'VSD_z061_Right.nii.gz'  -> ('VSD_z061', 'right')
    'Case8_PartRight.nii.gz' -> ('Case8', 'right')
    """
    stem = filename.replace(".nii.gz", "")
    if "_Part" in stem:                      # fractured
        case, sidepart = stem.split("_Part")
        side = "left" if "left" in sidepart.lower() else "right"
    else:                                     # healthy
        case, side_tok = stem.rsplit("_", 1)
        side = side_tok.lower()
    return case, side


def load_subject(nii_path: Path):
    """Load a normalized [0,1] knee volume, reverse the HU windowing, and build a DiffDRR Subject.

    predrr data is float [0,1]. DiffDRR's read() classifies voxels by HU thresholds (-800/350),
    so we reverse the [-450, 1050] windowing back to pseudo-HU first.
    """
    img = nib.load(str(nii_path))
    arr = img.get_fdata().astype(np.float32)          # [0,1]
    lo, hi = HU_WINDOW
    hu = arr * (hi - lo) + lo                          # pseudo-HU
    tensor = torch.from_numpy(hu)[None].float()        # (1, X, Y, Z)
    si = ScalarImage(tensor=tensor, affine=img.affine)
    return read(si, orientation="AP", bone_attenuation_multiplier=BONE_MULT)

In [ ]:
def make_drr(subject):
    """Construct the DiffDRR renderer for a subject."""
    return DRR(subject, sdd=SDD, height=HEIGHT, width=WIDTH, delx=DELX, dely=DELY).to(DEVICE)


def render_view(drr, view: str):
    """Render one view; returns the raw line-integral image as a (H, W) numpy array."""
    cfg = VIEWS[view]
    rot   = torch.tensor([cfg["rotation"]],    dtype=torch.float32, device=DEVICE)
    trans = torch.tensor([cfg["translation"]], dtype=torch.float32, device=DEVICE)
    out = drr(rot, trans, parameterization="euler_angles", convention="ZXY", degrees=True)
    return out.detach().cpu().numpy()[0, 0]


def postprocess(line_integral: np.ndarray) -> np.ndarray:
    """Beer-Lambert transform -> bone-bright -> per-image min-max normalize to [0,1] (float32)."""
    norm_li = line_integral / (line_integral.max() + 1e-8)
    x = np.exp(-MU * norm_li)        # Beer-Lambert: more attenuation -> darker
    x = 1.0 - x                      # invert: bone (high attenuation) becomes bright
    x = (x - x.min()) / (x.max() - x.min() + 1e-8)
    return x.astype(np.float32)

In [ ]:
def save_drr(image01: np.ndarray, dataset: str, case: str, side: str, view: str):
    """Save a normalized [0,1] DRR as 16-bit PNG + float32 .npy."""
    out_dir = DRR_OUT_DIR / dataset / case / side
    out_dir.mkdir(parents=True, exist_ok=True)
    png_path = out_dir / f"{view}.png"
    npy_path = out_dir / f"{view}.npy"
    imageio.imwrite(png_path, (image01 * 65535.0).round().astype(np.uint16))
    np.save(npy_path, image01)
    return png_path, npy_path


def generate_case(nii_path: Path, dataset: str, records: list):
    """Render + save AP and LAT for one knee file. Appends metadata rows to `records`."""
    case, side = parse_case_side(nii_path.name)
    subject = load_subject(nii_path)
    drr = make_drr(subject)
    images = {}
    for view in VIEWS:
        li = render_view(drr, view)
        img01 = postprocess(li)
        png_path, npy_path = save_drr(img01, dataset, case, side, view)
        images[view] = img01
        records.append({
            "case": case, "side": side, "dataset": dataset, "view": view,
            "sdd": SDD, "sod": SOD, "delx": DELX, "height": HEIGHT, "width": WIDTH,
            "bone_mult": BONE_MULT, "mu": MU, "li_max": float(li.max()),
            "png_path": str(png_path), "npy_path": str(npy_path),
        })
    return case, side, images

## Smoke test — single case before batch

Run one healthy and one fractured case to confirm GPU rendering works and outputs look correct
before committing to the full batch (~1.5 min on CPU per case; should be seconds on GPU).

**Check:** both AP and LAT render without error, bone is bright, full knee is in frame.

In [ ]:
SMOKE_FILES = [
    ("healthy",   "VSD_001_Left.nii.gz"),
    ("fractured", "Case1_PartLeft.nii.gz"),
]

smoke_records = []
smoke_results = []
for dataset, fname in SMOKE_FILES:
    logger.info(f"Smoke test: {dataset}/{fname}")
    case, side, images = generate_case(PREDRR_DIR / dataset / fname, dataset, smoke_records)
    smoke_results.append((dataset, case, side, images))

# Visual QA
fig, axes = plt.subplots(len(smoke_results), 2, figsize=(6, 3 * len(smoke_results)))
for row, (dataset, case, side, images) in enumerate(smoke_results):
    for col, view in enumerate(["ap", "lat"]):
        ax = axes[row, col]
        ax.imshow(images[view], cmap="gray")
        ax.set_title(f"{dataset}/{case}/{side} — {view.upper()}", fontsize=9)
        ax.axis("off")
plt.tight_layout()
plt.savefig(str(DRR_OUT_DIR / "smoke_test_qa.png"), dpi=100)
plt.show()
logger.info("Smoke test passed — proceeding to batch")

## Full batch processing

Renders all 58 healthy + 14 fractured files (72 total → 144 DRRs).
Skips files already rendered (re-run safe).
Writes `drr_generation_metadata.csv` on completion.

In [ ]:
assert ENV == "HPC", "Set ENV='HPC' in the config cell before running batch."

batch_records = []
for dataset in ["healthy", "fractured"]:
    files = sorted((PREDRR_DIR / dataset).glob("*.nii.gz"))
    logger.info(f"Batch {dataset}: {len(files)} files")
    for nii_path in tqdm(files, desc=f"DRR {dataset}"):
        case, side = parse_case_side(nii_path.name)
        # Skip if both views already rendered (re-run safe)
        ap_done  = (DRR_OUT_DIR / dataset / case / side / "ap.npy").exists()
        lat_done = (DRR_OUT_DIR / dataset / case / side / "lat.npy").exists()
        if ap_done and lat_done:
            logger.info(f"  skip {nii_path.name} (already rendered)")
            continue
        generate_case(nii_path, dataset, batch_records)

meta_path = DRR_OUT_DIR / "drr_generation_metadata.csv"
pd.DataFrame(batch_records).to_csv(meta_path, index=False)
logger.info(f"Batch complete: {len(batch_records)} new DRRs rendered")
logger.info(f"Metadata -> {meta_path}")

In [ ]:
# Verify output: count files and check one healthy + one fractured .npy
all_npy = list(DRR_OUT_DIR.rglob("*.npy"))
all_png = list(DRR_OUT_DIR.rglob("*.png"))
print(f"Total .npy files: {len(all_npy)} (expected 144)")
print(f"Total .png files: {len(all_png)} (expected 144)")

# Spot-check
for p in [
    DRR_OUT_DIR / "healthy"   / "VSD_001" / "left"  / "ap.npy",
    DRR_OUT_DIR / "fractured" / "Case1"   / "left"  / "ap.npy",
]:
    a = np.load(p)
    print(f"{p.relative_to(DRR_OUT_DIR)}: dtype={a.dtype} shape={a.shape} range=[{a.min():.3f}, {a.max():.3f}]")